In [ ]:
import pandas as pd
import re
import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import umap
from sklearn.datasets import load_digits


import hdbscan
import sklearn.cluster as cluster
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score


In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-05-11.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")


df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")
df_col

In [ ]:
df0[["q45_clé", "q24_research_fields"]].loc[df0.q24_research_fields.str.contains("LS6 Immunité, infection et immunothérapie")]

# Profil disciplinaire

In [ ]:
def split_multiple_choices(data, column, index, sep = '|'):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_split = data.copy()
    df_split[column]= df0.apply(lambda row: row[column].replace(";","|") ,1 )
    df_split[column] = df_split[column].str.split(sep)
    df_explode = df_split.explode(column)
    gb_data = df_explode.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100

    return gb_data, df_explode

In [ ]:
df0["q24_research_fields"]= df0.apply(lambda row: row.q24_research_fields.replace(";","|") ,1 )


In [ ]:
gb_data, df_exp = split_multiple_choices(df0, column='q24_research_fields', index = "q45_clé")


In [ ]:
df_rf = df_exp[["q45_clé", 'q24_research_fields']].copy()
df_rf.loc[df_rf.q24_research_fields.str.contains("SH"), "q24_research_domain"] = "Sciences sociales et humaines"
df_rf.loc[df_rf.q24_research_fields.str.contains("LS"), "q24_research_domain"] = "Sciences de la vie"
df_rf.loc[df_rf.q24_research_fields.str.contains("PE"), "q24_research_domain"] = "Sciences physiques et ingénierie"
df_rf["value"] = 1
df_rf.q24_research_fields.value_counts()
df_rf

# Clusterisation des individus

Après le regroupement des champs disciplinaires, nous allons utilisés les mêmes algorithmes pour, cette fois, identifier les individus aux profils disciplinaires proches. L'exploration des données a mis en évidence l'existence de chercheurs qui "cochent" toutes les disciplines ou qui ont répondu deux fois (sans sélectionner le même nombre de disciplines). Nous commençons par enlevés ces lignes qui créent du bruit.

Ensuite, nous avons commencé par une approche "non-supervisée", puis au fur et à mesure de la classification des individus, nous avons utilisés les labels pour contruire des modèles semi-supervisé. Au final, nous avons identifiés neuf aires disciplinaires. Ces informations sont contenues dans les fichiers "tableau_cluster_ind.txt" et "tableau_cluster.txt". Le premier groupe que nous avons identifié



### Unsupervised classification

In [ ]:
df_rfi = df_rf.pivot(index =['q45_clé'], columns= 'q24_research_fields', values = "value").fillna(0).reset_index()
m_row = df_rfi[df_rfi.columns[1:]].values



In [ ]:
fig, ax = plt.subplots(1, figsize=(10,10))



embedding = umap.UMAP(n_neighbors=5,
                      min_dist=0.1,
                      n_components=2,
                      metric='cosine', random_state =42).fit_transform(m_row)

sns.scatterplot(x=embedding[:,0], y=embedding[:,1],  ax=ax)


In [ ]:
labels = hdbscan.HDBSCAN(
    min_samples=4,
    min_cluster_size=4, gen_min_span_tree=True
).fit(embedding)

labels.single_linkage_tree_.plot()
print(len(set(labels.labels_)))
print(len([x for x in labels.labels_ if x == -1]))

HDBSCAn détecte 13 clusters (en comptant les outliers). Il y a 7 outliers. Nous allons examiner les résultats en les enregistant dans un fichier texte, plus facile à annoter. Pour nous aider à labelliser les points, nous allons ajouter des infor sur le labo, le nom et le prénom (permet de rechercher les info sur Internet).

Attention : les données ne sont donc plus pseudonymisées (faire cette partie en local)

In [ ]:
dfn = pd.read_csv("buparis8_chercheurs_besoins_accompagnement_5-11-2026_16_7.csv", sep =";")
dfn = dfn.rename(columns={"142. Nom :":"q42_nom", "143. Prénom :":"q42_prenom", "146. Clé":"q45_clé"})

affil = pd.read_csv("list_affiliation.csv", sep =",")
affil1 = affil.merge(dfn[["q45_clé","q42_nom","q42_prenom"]], on = ["q45_clé"], how = "left")

In [ ]:
df_rf.loc[df_rf.q45_clé=="HKN6-95PQ"]

In [ ]:
#%%capture cap
dict_clusters = {}

for n, x in enumerate(df_rfi.q45_clé):
    dict_clusters[x] = labels.labels_[n]


with open("tableau_codage_cluster.md", 'w') as fout: 
    for x in range(-1,len(set(labels.labels_))):
        fout.write(f"Cluster {x} :\n\n")
        name_column = df_rfi.columns[1:22]
        abrev = "-|".join([re.search(r"\w*\d+", x).group() for x in name_column ])
        table_frame = "|".join(["----" for col in name_column])
        fout.write(f"|{abrev}|key|nom|ufr_labo|\n")
        fout.write(f"|{table_frame}|----|----|---|\n")

        compteur = 0
        for v in dict_clusters:
            if dict_clusters[v] == x:
                compteur+=1
                dtmp0 = df_rfi.merge(affil1, on = "q45_clé", how = "left")
                nom = dtmp0[dtmp0.columns[27:]].loc[dtmp0.q45_clé == v].values
                ufr = dtmp0[dtmp0.columns[22]].loc[dtmp0.q45_clé == v].values
                dtmp1 = dtmp0[dtmp0.columns[1:22]].loc[dtmp0.q45_clé == v].values
                #fout.write(f'{v}: {",".join([str(x) for x in dtmp1[0]])} | )
                fout.write(f'|{"-|".join([str(x).replace("0.0","---") for x in dtmp1[0]])}|{v}|{" ".join([str(x) for x in nom[0]])} | {ufr[0]}| \n')
        fout.write("\n=====================\n")


Cluster 0 :

|key|LS4|LS5|LS6|LS7|LS8|LS9|PE1|PE10|PE5|PE6|PE7|PE8|PE9|SH1|SH2|SH3|SH4|SH5|SH6|SH7|SH8|nom|ufr_labo|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|572G-RBZC|--|--|--|--|--|--|--|--|--|--|--|--|--|--|--|**1.0**|--|--|--|--|--|nan nan | nan| 
|6454-S2QM|--|--|--|--|--|--|--|--|--|**1.0**|--|--|--|--|--|**1.0**|--|--|--|--|--|nan nan | nan| 
|E3YL-PHZV|--|--|--|--|--|--|--|--|--|--|--|--|--|--|--|**1.0**|--|--|--|--|--|nan nan | nan| 
|HKN6-95PQ|--|--|--|--|--|--|--|--|--|--|--|--|--|--|--|**1.0**|--|--|--|--|--|de Chalvron stephanie | nan| 
|MXD6-JC3F|--|--|--|--|--|--|--|--|--|--|--|--|--|--|--|**1.0**|--|--|--|--|--|nan nan | nan| 
|PQML-4NVQ|--|--|--|--|--|--|--|--|--|--|--|--|--|--|--|**1.0**|--|--|--|--|--|nan nan | nan| 
|R9R6-P7CN|--|--|--|--|--|--|--|--|--|--|--|--|--|--|--|**1.0**|--|--|--|--|--|nan nan | Culture et communication| 
|S3JS-37LL|--|--|--|--|--|--|--|--|--|--|--|--|--|--|--|**1.0**|--|--|--|--|--|Mollaret Patrick | Psychologie| 
|VTX9-R7JQ|--|--|--|--|--|--|--|--|--|--|--|--|--|--|--|**1.0**|--|--|--|--|--|Gonzalez Daniela | Culture et communication| 
|XA8V-2558|--|--|--|--|--|--|--|--|--|--|--|--|--|--|--|**1.0**|--|--|--|--|--|nan nan | nan| 

## Semisupervised dimension reduction

In [ ]:
#### df_rf5.to_csv("embedding_indiv_research_fields.csv", sep = ",", index = False)

dfn = pd.read_csv("buparis8_chercheurs_besoins_accompagnement_5-11-2026_16_7.csv", sep =";")
dfn = dfn.rename(columns={"142. Nom :":"q42_nom", "143. Prénom :":"q42_prenom", "146. Clé":"q45_clé"})

df_rf5 = pd.read_csv("embedding_indiv_research_fields.csv", sep =",")

affil = pd.read_csv("list_affiliation.csv", sep =",")
affil1 = affil.merge(dfn[["q45_clé","q42_nom","q42_prenom"]], on = ["q45_clé"], how = "left")
affil1

In [ ]:
with open("tableau_cluster_ind.txt", "r", newline='') as fin:
    lines = fin.readlines()

dict_domain = {}
dict_area = {}
list_domain = []
list_area = []
for l in lines :
    if len(l.split(':')) > 0 and l.count("Cluster") == 0:
        l_split = l.split(":")
        id_row = l_split[0]
        fields = l_split[-1]
        if len(fields.split(',')) > 1:
            area = fields.split(",")[-1].strip()
            domain = fields.split(",")[0].strip()
            dict_domain[id_row] = domain
            dict_area[id_row] = area
            if area not in list_area:
                list_area.append(area)
            else:
                pass
            if domain not in list_domain:
                list_domain.append(domain)
            else:
                pass
        else:
            pass
    else:
        pass

no_area = {}
for c in list_area:
    no_area[c] = list_area.index(c)



In [ ]:
df_rf6 = df_rf5.drop(columns=['cluster',
       'lab_cluster', 'new_clust', 'semi_cluster', 'semi_lab_cluster']).loc[~df_rf5.q45_clé.isin(["9EAP-NB4B","QNYZ-3MH2","H5X6-KL2Z",'PS8W-TJ9P'])] #

df_rf6["domain"] = df_rf6.q45_clé.map(dict_domain.get)

df_rf6["area"] = df_rf6.q45_clé.map(dict_area.get)
df_rf6["no_area"] = df_rf6.area.map(no_area.get)

df_rf6 = df_rf6.sort_values(by = ["area","domain"])

In [ ]:
df_rf5.to_csv("../data/clusterisation/research_field_ind_codage_intermediaire.csv", sep =",", index = False)
df_rf6.to_csv("../data/clusterisation/research_field_ind_VF.csv", sep =",", index = False)

Cluster : Environnement et espace

|id_row|SH1|SH2|SH3|SH5|SH6|SH7|SH8|PE7|PE8|PE9|PE10|LS4|LS6|LS7|LS8|LS9|SH4|LS5|PE6|PE1|PE5|research_domain|
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
|N99W-FKD9|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0 | Mobilité humaine environnement et espace 
|CALH-8CSE|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0 | Mobilité humaine environnement et espace 
|8NR6-Q3ED|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0 | Mobilité humaine environnement et espace 
|WYVU-F8DF|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0 | Mobilité humaine environnement et espace 
|NSK5-6STJ|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0 | Mobilité humaine environnement et espace 
|SZY2-JZTU|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0 | Mobilité humaine environnement et espace 
|A3S4-MJGK|0.0|0.0|**1.0**|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0 | Mobilité humaine environnement et espace 
|KBVW-94Y2|0.0|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|**1.0**|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0|0.0 | Mobilité humaine environnement et espace 

In [ ]:
df_rf6.loc[df_rf6["q45_clé"]=="KBVW-94Y2"].values

In [ ]:
#%%capture cap
dict_area = dict(zip(df_rf6.q45_clé, df_rf6.area))

with open("tableau_cluster.txt", 'w') as fout: 
    for x in list_area:
        fout.write(f"### {x}\n\n")
        name_column = df_rf6.columns[1:22]
        abrev = "|".join([re.search(r"\w*\d+", x).group() for x in name_column ])
        table_frame = "|".join(["---:" for col in name_column])
        fout.write(f"|{abrev}|\n")
        fout.write(f"|{table_frame}|\n")

        compteur = 0
        for v in dict_area:
            if dict_area[v] == x:
                compteur+=1
                dtmp0 = df_rf6.merge(affil, on = "q45_clé", how = "left")
                dtmp = dtmp0[dtmp0.columns[35]].loc[dtmp0.q45_clé == v].values
                dtmp1 = dtmp0[dtmp0.columns[1:22]].loc[dtmp0.q45_clé == v].values
                fout.write(f'|{"|".join([str(x).replace("1.0","**1.0**") for x in dtmp1[0]])}|\n')
        fout.write("\n\n")


In [ ]:
fig, ax = plt.subplots(1, figsize=(10,10))

matrix_ind = df_rf6[df_rf6.columns[1:22]].values
matrix_ind


embedding = umap.UMAP(n_neighbors=5,
                      min_dist=0.1,
                      n_components=2,
                      metric='cosine', random_state =42).fit_transform(matrix_ind, target = df_rf6[df_rf6.columns[-1]].values)

sns.scatterplot(x=embedding[:,0], y=embedding[:,1],  ax=ax, hue = [str(x) for x in df_rf6.area])


In [ ]:
dict_nom = dict(zip(affil1.q45_clé, affil1.q42_nom))
dict_prenom = dict(zip(affil1.q45_clé, affil1.q42_prenom))

list_row = []
for n, x in enumerate(embedding):
    id_key = df_rf6[df_rf6.columns[0]].values[n]
    try:
        nom = dict_nom[id_key].lower()
    except:
        nom = "not defined"
    try:
        prenom = dict_prenom[id_key].lower()
    except:
        prenom = "not defined"
    dict_row = {"q45_clé": id_key,
                      "x": x[0],
                      "y": x[1],
                      "nom": nom,
                      "prenom": prenom
                     }
    list_row.append(dict_row)
    

df_emb = pd.DataFrame.from_dict(list_row)
df_emb

In [ ]:
import plotly.express as px


fig_2d = px.scatter(
    df_emb, x="x", y="y",
    color=df_rf6.area, hover_data=['q45_clé', 'nom', 'prenom'],
)

fig_2d.write_html('plotly_embedding.html')

In [ ]:
labels = hdbscan.HDBSCAN(
    min_samples=4,
    min_cluster_size=4, gen_min_span_tree=True
).fit(embedding)

labels.single_linkage_tree_.plot()
print(len(set(labels.labels_)))
print(len([x for x in labels.labels_ if x == -1]))

In [ ]:
labels.minimum_spanning_tree_.plot(edge_cmap='viridis', 
                                      edge_alpha=0.6, 
                                      node_size=60, 
                                      edge_linewidth=2)


In [ ]:
name_column = df_rf6.columns[1:22]
name_column

for x in name_column:
    abrev = re.search(r"\w*\d+", x)


In [ ]:
#%%capture cap
dict_clusters = {}

for n, x in enumerate(df_rf6.q45_clé):
    dict_clusters[x] = labels.labels_[n]


with open("tableau_cluster.txt", 'w') as fout: 
    for x in range(-1,len(set(labels.labels_))):
        fout.write(f"Cluster : {x}\n\n")
        name_column = df_rf6.columns[1:22]
        abrev = ",".join([re.search(r"\w*\d+", x).group() for x in name_column ])
        fout.write(f"colmuns : {abrev}\n")
        compteur = 0
        for v in dict_clusters:
            if dict_clusters[v] == x:
                compteur+=1
                dtmp0 = df_rf6.merge(affil, on = "q45_clé", how = "left")
                dtmp = dtmp0[dtmp0.columns[35]].loc[dtmp0.q45_clé == v].values
                dtmp1 = dtmp0[dtmp0.columns[1:22]].loc[dtmp0.q45_clé == v].values
                fout.write(f'{v}: {",".join([str(x) for x in dtmp1[0]])} | {dtmp[0]} \n')
        fout.write("\n=====================\n")


In [ ]:
with open("tableau_cluster.txt", "r", newline='') as fin:
    lines = fin.readlines()

row_cluster = []
no_cluster = {}
for l in lines :
    if re.match("Cluster", l):
        #print(l)
        id_area = re.findall(r"\d+", l)
        area = l.split(":")[-1].strip()
        no_cluster[id_area[0]] = area
    else:
        if re.search("-",l):
            l_split = l.split(":")
            l_cluster = l_split[-1].split("|")
            #print(l_split[0], list_cluster)
            dict_row = {'q45_clé': l_split[0].strip(),
                       'no_area': id_area[0],
                       'area': area,
                       'fields': l_cluster[-1].strip()}
            row_cluster.append(dict_row)
         

df_cluster = pd.DataFrame.from_dict(row_cluster)

In [ ]:
df_emb1  = df_emb.merge(df_cluster, on = ["q45_clé"], how ="left")
list_point = [x for x in df_emb.q45_clé.loc[df_emb.new_cluster==-1]]
df_rf6[df_rf6.columns[0:23]].loc[df_rf6.q45_clé.isin(list_point)]

In [ ]:

fig_2d = px.scatter(
    df_emb1, x="x", y="y",
    color="area", hover_data=['q45_clé', 'nom', 'prenom'],
)

fig_2d.write_html('plotly_embedding2.html')